# 06 · Aprendizaje automático con MLlib

**Objetivos de aprendizaje**

- Preparar variables con `Imputer`, `StringIndexer`, `OneHotEncoder` y `VectorAssembler`.
- Construir un `Pipeline` de regresión o de clasificación según el tipo de variable objetivo.
- Evaluar el modelo frente a una línea base y analizar la importancia de las variables.
- Reconocer la fuga de datos (*data leakage*).
- Segmentar registros con K-Means y elegir el número de grupos con la silueta.

| Aspecto | Detalle |
|---|---|
| Duración estimada | 45 minutos |
| Requisito previo | Cuaderno 04 (recomendado) |
| Entrada | Dataset activo (versión limpia, si existe) |
| Salida | `data/procesado/modelos/` |

### Cómo utilizar este cuaderno

1. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con `Shift + Enter`. Cada celda puede depender de variables creadas en celdas anteriores.
2. Las celdas que comienzan con `# >>> CELDA EDITABLE` son las únicas que deben modificarse para adaptar el análisis a otro dataset. El resto del código se ajusta automáticamente a las columnas detectadas.
3. Cada sección sigue la misma estructura: **fundamento** (qué es y por qué se hace), **código**, **interpretación** del resultado y **adaptación** al dataset propio.
4. Ante un error `NameError` (variable no definida), lo habitual es que no se haya ejecutado una celda anterior. La solución es *Kernel > Restart Kernel and Run All Cells*.
5. Mientras la sesión de Spark está activa, la Spark UI (normalmente <http://localhost:4040>) muestra los *jobs*, *stages* y tareas ejecutados.

La guía completa de uso de los cuadernos se encuentra en `docs/08-uso-de-notebooks.md`.

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("06_MLlib")

## Carga del dataset activo

El dataset activo es el que se configuró en el cuaderno `01_configuracion_dataset.ipynb` y quedó guardado en `data/dataset_config.json`. Si no se ha configurado ninguno, se utiliza el dataset de ejemplo (`data/ejemplo/ventas_tienda.csv`).

La función `clasificar_columnas()` asigna a cada columna un rol, que los cuadernos usan para decidir qué análisis aplicar:

| Variable | Rol | Criterio |
|---|---|---|
| `NUM` | numéricas | tipos numéricos que no son identificadores |
| `CAT` | categóricas | texto o booleanos con pocos valores distintos (hasta `max_categorias`) |
| `FEC` | fechas | tipos `date` o `timestamp` |
| `TXT` | texto | texto con muchos valores distintos (comentarios, descripciones) |
| — | identificadores | columnas excluidas en la configuración o con valores casi únicos |

Con un dataset propio, conviene revisar esta clasificación. Si una columna quedó en un rol inadecuado, puede corregirse en la configuración (`columnas_excluir`, `max_categorias`) o directamente en las listas, por ejemplo `CAT.remove("columna")`.

In [ ]:
# Carga del dataset activo (definido y guardado en el cuaderno 01)
df, cfg = cargar_desde_config(spark, version="auto")

# Clasificación automática de columnas por rol analítico
roles = clasificar_columnas(df, excluir=cfg["columnas_excluir"], max_categorias=cfg["max_categorias"])
NUM, CAT, FEC, TXT = roles["numericas"], roles["categoricas"], roles["fechas"], roles["texto"]
objetivo = columna_objetivo(cfg, df)
resumen_roles(roles)
print("Columna objetivo:", objetivo)

## 1. Fundamento: MLlib y los pipelines

MLlib es la biblioteca de aprendizaje automático distribuido de Spark (Meng et al., 2016). Su API se organiza en tres conceptos:

- **Transformer**: transforma un DataFrame en otro (por ejemplo, un modelo entrenado que agrega predicciones).
- **Estimator**: aprende de los datos y produce un Transformer (por ejemplo, un algoritmo que se entrena con `fit()`).
- **Pipeline**: secuencia de etapas que se entrena y aplica como una sola unidad, lo que garantiza que el mismo preprocesamiento se aplique al entrenamiento y a la evaluación (ASF, 2026c).

El flujo de este cuaderno es:

```
numéricas  ──► Imputer (mediana) ─────────────────────────┐
                                                          ├─► VectorAssembler ─► modelo
categóricas ─► StringIndexer ─► OneHotEncoder ────────────┘
```

## 2. Variable objetivo y tipo de problema

- Si la variable objetivo es numérica con muchos valores distintos, el problema es de **regresión** (predecir un número).
- Si es categórica o numérica con pocos valores, el problema es de **clasificación** (predecir una categoría).

**Adaptación.** La variable objetivo proviene de la configuración (`columna_objetivo`). Puede cambiarse en la celda siguiente. `EXCLUIR_DEL_MODELO` permite retirar variables explicativas, por ejemplo las que se calculan a partir del objetivo.

In [ ]:
# >>> CELDA EDITABLE: variable objetivo y variables excluidas
OBJETIVO = objetivo                  # p. ej. "total", o el nombre de una columna categórica
EXCLUIR_DEL_MODELO = []              # p. ej. ["cantidad", "precio_unitario"]

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

if OBJETIVO is None:
    print("No se definió variable objetivo: se omite la Parte A y se continúa con K-Means (Parte B).")
else:
    tipo_obj = dict(df.dtypes)[OBJETIVO]
    distintos_obj = df.select(OBJETIVO).distinct().count()
    es_regresion = tipo_obj.split("(")[0] in TIPOS_NUMERICOS and distintos_obj > 15
    num_feats = [c for c in NUM if c not in (OBJETIVO, *EXCLUIR_DEL_MODELO)]
    cat_feats = [c for c in CAT if c not in (OBJETIVO, *EXCLUIR_DEL_MODELO)]
    print("Tipo de problema       :", "regresión" if es_regresion else f"clasificación ({distintos_obj} clases)")
    print("Variables numéricas    :", num_feats)
    print("Variables categóricas  :", cat_feats)

## Parte A · Modelo supervisado

### 3. Preparación de los datos

- Se eliminan las filas sin valor en la variable objetivo.
- Las variables numéricas se convierten a `double`, requisito del `Imputer`.
- Las variables categóricas (y el objetivo, en clasificación) se convierten a texto, para que `StringIndexer` asigne un índice a cada categoría; así se admiten también columnas booleanas.
- Los datos se dividen en entrenamiento (80 %) y prueba (20 %). La semilla (`seed=42`) hace que la división sea reproducible.

In [ ]:
if OBJETIVO:
    datos = df.dropna(subset=[OBJETIVO])
    for c in num_feats:
        datos = datos.withColumn(c, F.col(c).cast("double"))
    for c in cat_feats:
        datos = datos.withColumn(c, F.col(c).cast("string"))
    if es_regresion:
        datos = datos.withColumn("label", F.col(OBJETIVO).cast("double"))
    else:
        datos = datos.withColumn(OBJETIVO, F.col(OBJETIVO).cast("string"))   # admite objetivos booleanos

    entrenamiento, prueba = datos.randomSplit([0.8, 0.2], seed=42)
    entrenamiento.cache()
    print(f"Entrenamiento: {entrenamiento.count():,} | Prueba: {prueba.count():,}")

### 4. Construcción del pipeline

Se utiliza un **bosque aleatorio** (*random forest*): un conjunto de árboles de decisión entrenados sobre muestras distintas de los datos, cuyas predicciones se promedian (regresión) o se votan (clasificación). Es robusto, admite relaciones no lineales y estima la importancia de cada variable (Breiman, 2001).

`handleInvalid="keep"` evita errores cuando en la prueba aparece una categoría que no estaba en el entrenamiento.

In [ ]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.classification import RandomForestClassifier

if OBJETIVO and (num_feats or cat_feats):
    etapas = []
    entradas = []
    if num_feats:
        etapas.append(Imputer(inputCols=num_feats, outputCols=[f"{c}_imp" for c in num_feats], strategy="median"))
        entradas += [f"{c}_imp" for c in num_feats]
    if cat_feats:
        etapas += [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_feats]
        etapas.append(OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_feats],
                                    outputCols=[f"{c}_ohe" for c in cat_feats], handleInvalid="keep"))
        entradas += [f"{c}_ohe" for c in cat_feats]
    etapas.append(VectorAssembler(inputCols=entradas, outputCol="features", handleInvalid="keep"))

    if es_regresion:
        etapas.append(RandomForestRegressor(featuresCol="features", labelCol="label",
                                            numTrees=50, maxDepth=8, seed=42))
    else:
        etapas.insert(0, StringIndexer(inputCol=OBJETIVO, outputCol="label", handleInvalid="skip"))
        etapas.append(RandomForestClassifier(featuresCol="features", labelCol="label",
                                             numTrees=50, maxDepth=8, seed=42))
    pipeline = Pipeline(stages=etapas)
    print("Etapas del pipeline:", [type(e).__name__ for e in etapas])
elif OBJETIVO:
    print("No hay variables explicativas disponibles para el modelo.")

### 5. Entrenamiento y predicción

`fit()` entrena todas las etapas en orden sobre los datos de entrenamiento; `transform()` aplica el pipeline entrenado a los datos de prueba y agrega la columna `prediction`.

In [ ]:
if OBJETIVO and (num_feats or cat_feats):
    import time
    t = time.time()
    modelo = pipeline.fit(entrenamiento)
    print(f"Entrenamiento completado en {time.time() - t:.1f} s")
    predicciones = modelo.transform(prueba)
    predicciones.select(OBJETIVO, "label", "prediction").show(8)

### 6. Evaluación frente a una línea base

Una métrica aislada no indica si el modelo es útil. Por eso se compara con una **línea base** ingenua:

- **Regresión**: predecir siempre el promedio del entrenamiento. Métricas: RMSE (error cuadrático medio, en las unidades del objetivo), MAE (error absoluto medio) y R² (proporción de la variabilidad explicada; 1 es perfecto, 0 equivale a la línea base).
- **Clasificación**: predecir siempre la clase más frecuente. Métricas: exactitud (*accuracy*) y F1 ponderado.

In [ ]:
import pandas as pd
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator

if OBJETIVO and (num_feats or cat_feats):
    if es_regresion:
        # Línea base: predecir siempre el promedio del conjunto de entrenamiento
        media = entrenamiento.agg(F.avg("label")).first()[0]
        base = predicciones.withColumn("prediction", F.lit(media))
        evaluador, nombres_metricas = RegressionEvaluator, ["rmse", "mae", "r2"]
    else:
        # Línea base: predecir siempre la clase más frecuente. StringIndexer asigna el índice 0.0
        # a la clase más frecuente del entrenamiento, por lo que basta con predecir 0.0.
        base = predicciones.withColumn("prediction", F.lit(0.0))
        evaluador, nombres_metricas = MulticlassClassificationEvaluator, ["accuracy", "f1"]
    filas = []
    for m in nombres_metricas:
        ev = evaluador(labelCol="label", predictionCol="prediction", metricName=m)
        filas.append((m, ev.evaluate(predicciones), ev.evaluate(base)))
    metricas = pd.DataFrame(filas, columns=["métrica", "modelo", "línea_base"]).round(4)
    display(metricas)

**Interpretación.** El modelo es útil solo si mejora claramente la línea base: un RMSE menor o un R² mayor en regresión, o una exactitud mayor en clasificación.

Con el dataset de ejemplo y el objetivo `total`, el R² es muy alto. La razón es que `total` se calcula a partir de `cantidad`, `precio_unitario` y `descuento`: el modelo "aprende" una fórmula que ya está en los datos. Este fenómeno se denomina **fuga de datos** (*data leakage*): una variable explicativa contiene información que no estaría disponible al momento de predecir. El ejercicio 1 propone repetir el modelo excluyendo esas variables.

En clasificación, la matriz de confusión muestra cuántos registros de cada clase real (`label`) se predijeron en cada clase.

In [ ]:
if OBJETIVO and (num_feats or cat_feats) and not es_regresion:
    predicciones.groupBy("label").pivot("prediction").count().orderBy("label").show()

### 7. Importancia de las variables

El bosque aleatorio estima cuánto contribuye cada variable a reducir el error (Breiman, 2001). Los nombres se recuperan de los metadatos que `VectorAssembler` agrega a la columna `features`; las variables categóricas aparecen separadas por categoría (`columna_ohe_valor`).

In [ ]:
import pandas as pd

if OBJETIVO and (num_feats or cat_feats):
    meta = predicciones.schema["features"].metadata["ml_attr"]["attrs"]
    nombres = sorted([a for grupo in meta.values() for a in grupo], key=lambda a: a["idx"])
    importancias = modelo.stages[-1].featureImportances.toArray()
    display(pd.DataFrame({"variable": [a["name"] for a in nombres], "importancia": importancias})
            .sort_values("importancia", ascending=False).head(12).round(4).reset_index(drop=True))

### 8. Guardar y recuperar el modelo

Un `PipelineModel` guardado incluye todas las etapas de preprocesamiento, de modo que puede aplicarse directamente a datos nuevos con el mismo esquema.

In [ ]:
from pyspark.ml import PipelineModel

if OBJETIVO and (num_feats or cat_feats):
    ruta_modelo = str(ruta_absoluta(f"data/procesado/modelos/{cfg['nombre']}_{OBJETIVO}"))
    modelo.write().overwrite().save(ruta_modelo)
    recuperado = PipelineModel.load(ruta_modelo)
    recuperado.transform(prueba).select(OBJETIVO, "prediction").show(3)
    print("Modelo guardado en", ruta_modelo)

## Parte B · Agrupamiento no supervisado con K-Means

K-Means divide los registros en *k* grupos, de modo que cada registro pertenezca al grupo con el centro más cercano. No requiere variable objetivo. Como las distancias dependen de la escala, las variables se estandarizan antes (media 0 y desviación 1) con `StandardScaler`.

Para elegir *k* se usa el **coeficiente de silueta**, que compara la distancia de cada punto a su grupo con la distancia al grupo vecino más cercano; valores cercanos a 1 indican grupos bien separados (Rousseeuw, 1987).

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# >>> CELDA EDITABLE: variables para el agrupamiento
VARIABLES_KMEANS = [c for c in NUM if c not in EXCLUIR_DEL_MODELO][:6]
print("Variables:", VARIABLES_KMEANS)

In [ ]:
if len(VARIABLES_KMEANS) >= 2:
    base_k = df.select([F.col(c).cast("double").alias(c) for c in VARIABLES_KMEANS])
    prep = Pipeline(stages=[
        Imputer(inputCols=VARIABLES_KMEANS, outputCols=VARIABLES_KMEANS, strategy="median"),
        VectorAssembler(inputCols=VARIABLES_KMEANS, outputCol="v"),
        StandardScaler(inputCol="v", outputCol="features", withMean=True, withStd=True),
    ]).fit(base_k)
    escalado = prep.transform(base_k).cache()

    evaluador = ClusteringEvaluator()
    siluetas = {}
    for k in range(2, 7):
        siluetas[k] = evaluador.evaluate(KMeans(k=k, seed=42).fit(escalado).transform(escalado))
        print(f"k = {k}  silueta = {siluetas[k]:.3f}")
    k_optimo = max(siluetas, key=siluetas.get)
    print("k con mayor silueta:", k_optimo)
else:
    print("Se requieren al menos dos variables numéricas para K-Means.")

In [ ]:
if len(VARIABLES_KMEANS) >= 2:
    grupos = KMeans(k=k_optimo, seed=42).fit(escalado).transform(escalado)
    (grupos.groupBy("prediction")
       .agg(F.count("*").alias("registros"), *[F.round(F.avg(c), 2).alias(c) for c in VARIABLES_KMEANS])
       .orderBy("prediction").show(truncate=False))

**Interpretación.** La tabla muestra el promedio de cada variable por grupo, en sus unidades originales. Describir cada grupo con palabras (por ejemplo, "compras pequeñas y frecuentes") es el paso que convierte el resultado en conocimiento útil.

El gráfico siguiente proyecta los grupos sobre las dos primeras variables, con una muestra de hasta 5 000 registros.

In [ ]:
import matplotlib.pyplot as plt

if len(VARIABLES_KMEANS) >= 2:
    x, y = VARIABLES_KMEANS[:2]
    muestra = grupos.select(x, y, "prediction").limit(5000).toPandas()
    plt.figure(figsize=(7, 5))
    plt.scatter(muestra[x], muestra[y], c=muestra["prediction"], s=10, cmap="viridis")
    plt.xlabel(x); plt.ylabel(y); plt.title(f"Grupos K-Means (k = {k_optimo})")
    plt.colorbar(label="grupo"); plt.show()

## Ejercicios propuestos

1. Agregar a `EXCLUIR_DEL_MODELO` las variables con las que se calcula el objetivo (en el ejemplo, `cantidad`, `precio_unitario` y `descuento`), volver a entrenar y comparar las métricas. Explicar el resultado en términos de fuga de datos.
2. Cambiar la variable objetivo por una categórica (en el ejemplo, `segmento_cliente` o `canal`) y analizar si el modelo supera la línea base.
3. Reemplazar el bosque aleatorio por `LinearRegression` o `LogisticRegression` y comparar.
4. Asignar un nombre descriptivo a cada grupo de K-Means según sus promedios.

In [ ]:
spark.stop()

## Referencias

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

Meng, X., Bradley, J., Yavuz, B., Sparks, E., Venkataraman, S., Liu, D., Freeman, J., Tsai, D. B., Amde, M., Owen, S., Xin, D., Xin, R., Franklin, M. J., Zadeh, R., Zaharia, M., & Talwalkar, A. (2016). MLlib: Machine learning in Apache Spark. *Journal of Machine Learning Research, 17*(34), 1–7. https://jmlr.org/papers/v17/15-237.html

Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7

The Apache Software Foundation. (2026c). *Machine learning library (MLlib) guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/ml-guide.html